# Cleaning the GGDC Productivity Data: Aggregating to Three Broad Sectors

This notebook cleans `data/Global-Productivity-Sectoral-Database.dta` (a
country–sector–year panel, 103 countries, 1950–2017, GGDC-style 9 detailed
sub-sectors) and aggregates it down to the **three broad sectors — Agriculture,
Manufacturing, Services — that the McMillan–Rodrik decomposition (methodology doc
§6.3) is defined over**, restricted to the project's 24 African countries.

**The aggregation rule follows Herrendorf, Rogerson & Valentinyi (2014), "Growth and
Structural Transformation"** (NBER Working Paper 18996 / Handbook of Economic Growth,
Vol. 2B, Ch. 6) — confirmed directly from their own Data Appendix (pp. 95–96) rather
than assumed, and confirmed specifically for the data source this project uses: the
appendix entry immediately following their list of countries sourced from the
"Groningen Growth and Development Centre 10-sector Database" (their primary
historical data source, per their own §1) reads:

> 1. Agriculture corresponds to the sum of International Standard Industrial
> Classification (ISIC) sections A–B.
> 2. Manufacturing corresponds to the sum of ISIC sections C, D, F and includes
> mining, manufacturing, and construction.
> 3. Services correspond to the sum of ISIC sections E, G–P and include utilities,
> wholesale, retail trade, hotels and restaurants, transport, storage and
> communication, finance, insurance, real estate, business services, and community
> social and personal services.

Their own footnote 3 explains the terminology: *"We follow much of the literature and
use the term manufacturing in this context to refer to all activity that falls
outside of agriculture and services."* The one detail easy to get wrong: **utilities
(electricity, gas, water) go with Services, not Manufacturing**, under this specific
(ISIC-code-based) rule — some of the paper's other, non-GGDC data sources place
utilities with manufacturing instead, but this is the rule tied to the actual database
this project uses.

**What this notebook does, in order:**

1. Load the raw file and confirm its structure (country/sector/year panel, 9
   sub-sectors + a `Total` row).
2. Restrict to the project's 24 African countries, via a name→ISO3 crosswalk (the
   source keys countries by name, not ISO3).
3. Validate that `Total` really does equal the sum of the 9 sub-sectors, before
   trusting any further aggregation logic built on top of it.
4. Apply the Herrendorf–Rogerson–Valentinyi mapping to collapse the 9 sub-sectors
   into Agriculture / Manufacturing / Services, catching the same all-NaN-sum pandas
   pitfall documented in notebook 02.
5. Recompute labor productivity for each broad sector — **not** by averaging the
   sub-sector productivities, but as aggregated real value added over aggregated
   employment, exactly the way the source's own `Total` row is built. Do the same for
   the PPP-based measure, which requires backing out an implied PPP value-added
   series first since the source only reports the productivity ratio, not its
   PPP-terms numerator.
6. Compute employment shares.
7. Validate the whole pipeline by reconstructing `Total` from the three broad sectors
   and checking it matches the source's own `Total` row exactly.
8. Save the result.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

In [2]:
REPO_ROOT = Path("..")
GGDC_DTA = REPO_ROOT / "data" / "Global-Productivity-Sectoral-Database.dta"
OUT_DIR = REPO_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GGDC_DTA

WindowsPath('../data/Global-Productivity-Sectoral-Database.dta')

## 2. Load and inspect the raw panel

In [3]:
raw = pd.read_stata(GGDC_DTA)
print(f"{raw.shape[0]:,} rows, {raw['country'].nunique()} countries, "
      f"{int(raw['year'].min())}-{int(raw['year'].max())}")
print()
print("Sectors:", raw["sector"].cat.categories.tolist() if hasattr(raw["sector"], "cat")
      else sorted(raw["sector"].unique().tolist()))
raw.head()

40,040 rows, 103 countries, 1950-2017

Sectors: ['1.Agriculture', '2.Mining', '3.Manufacturing', '4.Utilities', '5.Construction', '6.Trade services', '7.Transport services', '8.Finance amd business services', '9.Other services', 'Total']


,country,sector,year,Value_added_nominal,Value_added_real,Employment,Labor_productivity_real,Labor_productivity_PPP
0,Angola,Total,2002.0,664206.7500,4.033341e+06,5471.140137,737.202942,7.888702
1,Angola,1.Agriculture,2002.0,38855.4896,2.617703e+05,2022.175000,129.449890,1.248572
2,Angola,2.Mining,2002.0,310204.5399,1.558314e+06,60.773000,25641.554688,331.678680
3,Angola,3.Manufacturing,2002.0,24365.2387,1.992284e+05,168.209000,1184.410034,9.412425
4,Angola,4.Utilities,2002.0,2580.9758,1.675762e+04,17.595000,952.407898,9.531796


## 3. Restrict to the project's 24 African countries

The source keys countries by name, not ISO3, so this needs an explicit crosswalk —
built by hand rather than fuzzy-matched, the same discipline notebooks 01–02 used for
ISO3 joins. One name doesn't match the obvious pattern: Tanzania is listed under its
formal name, "United Republic of Tanzania".

In [4]:
AFRICAN = {
    "DZA": "Algeria", "AGO": "Angola", "BWA": "Botswana", "BFA": "Burkina Faso",
    "CMR": "Cameroon", "EGY": "Egypt", "SWZ": "Eswatini", "ETH": "Ethiopia",
    "GHA": "Ghana", "KEN": "Kenya", "LSO": "Lesotho", "MWI": "Malawi",
    "MUS": "Mauritius", "MAR": "Morocco", "MOZ": "Mozambique", "NAM": "Namibia",
    "NGA": "Nigeria", "RWA": "Rwanda", "SEN": "Senegal", "SLE": "Sierra Leone",
    "ZAF": "South Africa", "TZA": "United Republic of Tanzania", "UGA": "Uganda",
    "ZMB": "Zambia",
}
NAME_TO_ISO3 = {v: k for k, v in AFRICAN.items()}

present = set(raw["country"].unique())
missing = [name for name in AFRICAN.values() if name not in present]
print(f"{len(AFRICAN)} African countries; names not found in the source (should be "
      f"empty): {missing}")

af = raw[raw["country"].isin(AFRICAN.values())].copy()
af["iso3"] = af["country"].map(NAME_TO_ISO3)
print(f"\n{len(af):,} rows for the African subset")
af.groupby("iso3")["year"].agg(["min", "max", "count"])

24 African countries; names not found in the source (should be empty): []

11,090 rows for the African subset


,min,max,count
iso3,,,
AGO,2002.0,2017.0,160
BFA,1970.0,2017.0,480
BWA,1964.0,2017.0,540
CMR,1965.0,2017.0,530
DZA,1999.0,2017.0,190
EGY,1960.0,2017.0,580
ETH,1961.0,2017.0,570
GHA,1960.0,2017.0,580
KEN,1969.0,2017.0,490


Coverage varies a lot by country — Algeria only starts in 1999, Sierra Leone in
2001, most others in the 1960s–1970s. This is a real data-availability constraint
inherited from the source, not something this notebook can fix; carry it forward
rather than pad it.

## 4. Validate: does `Total` equal the sum of the 9 sub-sectors?

Confirm this before trusting any aggregation built on top of it — the same
"diagnose, don't assume" discipline used throughout notebooks 01–02.

In [5]:
SUBSECTORS = [
    "1.Agriculture", "2.Mining", "3.Manufacturing", "4.Utilities", "5.Construction",
    "6.Trade services", "7.Transport services", "8.Finance amd business services",
    "9.Other services",
]

check = af[af["sector"].isin(SUBSECTORS)].groupby(["iso3", "year"], as_index=False)[
    ["Value_added_real", "Employment"]
].sum(min_count=1)
tot = af[af["sector"] == "Total"][["iso3", "year", "Value_added_real", "Employment"]]

merged = check.merge(tot, on=["iso3", "year"], suffixes=("_sumsub", "_total"))
va_diff = (merged["Value_added_real_sumsub"] - merged["Value_added_real_total"]).abs()
emp_diff = (merged["Employment_sumsub"] - merged["Employment_total"]).abs()
va_rel = va_diff / merged["Value_added_real_total"].abs()
emp_rel = emp_diff / merged["Employment_total"].abs()
note = ("expect ~1e-7, i.e. float32 storage precision -- Value_added_real/Employment "
        "are stored as float32, so a country with value added in the tens of millions "
        "can show an absolute gap of a few units purely from rounding, not a real "
        "data problem")
print(f"Max absolute diff -- value added: {va_diff.max():.6f}, employment: {emp_diff.max():.6f}")
print(f"Max relative diff -- value added: {va_rel.max():.2e}, employment: {emp_rel.max():.2e} ({note})")

Max absolute diff -- value added: 4.000000, employment: 0.001949
Max relative diff -- value added: 1.16e-07, employment: 5.74e-08 (expect ~1e-7, i.e. float32 storage precision -- Value_added_real/Employment are stored as float32, so a country with value added in the tens of millions can show an absolute gap of a few units purely from rounding, not a real data problem)


## 5. Check missingness before aggregating

A naive sum across sub-sectors would need to distinguish "this sub-sector is
genuinely missing" from "this sub-sector is genuinely zero" — the same pandas
all-NaN-sum pitfall notebook 02 caught building the trade-exposure variables. Check
which country-years are affected before deciding how to handle it.

In [6]:
missing_rows = af[af["sector"].isin(SUBSECTORS) & af["Value_added_real"].isna()]
print("Country-years with at least one missing sub-sector value:")
print(missing_rows.groupby("iso3")["year"].agg(["min", "max", "nunique"]))

Country-years with at least one missing sub-sector value:
         min     max  nunique
iso3                         
BWA   1964.0  1967.0        4
NAM   1960.0  1964.0        5


Only Namibia and Botswana are affected, in a handful of early years each. These
stay as `NaN` in the aggregated broad-sector output below (via `min_count=1`, the same
fix used in notebook 02) rather than silently becoming `0` — a real "we don't know"
is preserved as `NaN`, not misrepresented as "zero activity that year."

## 6. Apply the Herrendorf–Rogerson–Valentinyi mapping

In [7]:
SECTOR_MAP = {
    "1.Agriculture": "Agriculture",
    "2.Mining": "Manufacturing",
    "3.Manufacturing": "Manufacturing",
    "5.Construction": "Manufacturing",
    "4.Utilities": "Services",
    "6.Trade services": "Services",
    "7.Transport services": "Services",
    "8.Finance amd business services": "Services",
    "9.Other services": "Services",
}
BROAD_SECTORS = ["Agriculture", "Manufacturing", "Services"]

sub = af[af["sector"].isin(SUBSECTORS)].copy()
sub["broad_sector"] = sub["sector"].map(SECTOR_MAP)

# Back out an implied PPP value-added series -- the source only gives the PPP
# productivity ratio, not its numerator, so this recovers it from the definition
# (productivity = value added / employment) before summing across sub-sectors.
sub["implied_va_ppp"] = sub["Labor_productivity_PPP"] * sub["Employment"]

broad = sub.groupby(["iso3", "country", "broad_sector", "year"], as_index=False).agg(
    value_added_real=("Value_added_real", lambda s: s.sum(min_count=1)),
    employment=("Employment", lambda s: s.sum(min_count=1)),
    implied_va_ppp=("implied_va_ppp", lambda s: s.sum(min_count=1)),
)
broad["labor_productivity_real"] = broad["value_added_real"] / broad["employment"]
broad["labor_productivity_ppp"] = broad["implied_va_ppp"] / broad["employment"]
broad = broad.drop(columns=["implied_va_ppp"])

print(f"{len(broad):,} rows: {broad['iso3'].nunique()} countries x "
      f"{broad['broad_sector'].nunique()} broad sectors x up to "
      f"{broad['year'].nunique()} years")
broad.head(9)

3,327 rows: 24 countries x 3 broad sectors x up to 58 years


,iso3,country,broad_sector,year,value_added_real,employment,labor_productivity_real,labor_productivity_ppp
0,AGO,Angola,Agriculture,2002.0,261770.34375,2022.175,129.449896,1.248572
1,AGO,Angola,Agriculture,2003.0,282771.75000,2070.820,136.550618,1.384714
2,AGO,Angola,Agriculture,2004.0,308427.90625,2135.687,144.416249,1.662547
3,AGO,Angola,Agriculture,2005.0,322458.71875,2448.782,131.681268,1.557592
4,AGO,Angola,Agriculture,2006.0,375029.50000,2748.234,136.461997,1.843702
5,AGO,Angola,Agriculture,2007.0,395946.12500,3101.016,127.682709,1.852409
6,AGO,Angola,Agriculture,2008.0,414765.31250,3491.457,118.794335,1.814551
7,AGO,Angola,Agriculture,2009.0,438542.28125,3792.035,115.648268,1.821171
8,AGO,Angola,Agriculture,2010.0,475984.65625,4243.954,112.155941,2.005313


## 7. Employment shares

Each broad sector's share of that country-year's total employment across all three
broad sectors (equivalently, across all 9 original sub-sectors).

In [8]:
emp_totals = broad.groupby(["iso3", "year"])["employment"].transform(
    lambda s: s.sum(min_count=1)
)
broad["employment_share"] = broad["employment"] / emp_totals

print(f"employment_share non-null: {broad['employment_share'].notna().mean():.1%}")
print(f"Bounds: [{broad['employment_share'].min():.3f}, "
      f"{broad['employment_share'].max():.3f}]")

# Sanity check: shares within a country-year should sum to 1
share_sums = broad.groupby(["iso3", "year"])["employment_share"].sum()
print(f"Share sums -- min: {share_sums.min():.6f}, max: {share_sums.max():.6f} "
      f"(expect 1.0 wherever no sub-sector is missing)")

employment_share non-null: 99.5%
Bounds: [0.006, 1.000]
Share sums -- min: 1.000000, max: 1.000000 (expect 1.0 wherever no sub-sector is missing)


## 8. Validate against the source's own `Total` row

Reconstruct each country-year's economy-wide value added, employment, and
productivity from the three broad sectors, and check it against the source's `Total`
row exactly — this confirms the Herrendorf–Rogerson–Valentinyi mapping is a genuine
partition of the 9 sub-sectors (nothing double-counted, nothing dropped), not just
that the code runs.

In [9]:
recon = broad.groupby(["iso3", "year"], as_index=False).agg(
    value_added_real=("value_added_real", lambda s: s.sum(min_count=1)),
    employment=("employment", lambda s: s.sum(min_count=1)),
)
recon["labor_productivity_real"] = recon["value_added_real"] / recon["employment"]

tot2 = af[af["sector"] == "Total"][
    ["iso3", "year", "Value_added_real", "Employment", "Labor_productivity_real"]
]
check2 = recon.merge(tot2, on=["iso3", "year"], suffixes=("_recon", "_source"))
va_diff = (check2["value_added_real"] - check2["Value_added_real"]).abs()
emp_diff = (check2["employment"] - check2["Employment"]).abs()
prod_diff = (check2["labor_productivity_real"] - check2["Labor_productivity_real"]).abs()
va_rel = va_diff / check2["Value_added_real"].abs()
emp_rel = emp_diff / check2["Employment"].abs()
prod_rel = prod_diff / check2["Labor_productivity_real"].abs()
print(f"Max absolute error -- value added: {va_diff.max():.6f}, employment: {emp_diff.max():.6f}, "
      f"labor productivity: {prod_diff.max():.6f}")
print(f"Max relative error -- value added: {va_rel.max():.2e}, employment: {emp_rel.max():.2e}, "
      f"labor productivity: {prod_rel.max():.2e} (expect ~1e-7, float32 storage precision "
      f"-- see the same note in section 4; nothing double-counted or dropped in the sector mapping)")

Max absolute error -- value added: 8.000000, employment: 0.001949, labor productivity: 0.000482
Max relative error -- value added: 1.28e-07, employment: 5.74e-08, labor productivity: 1.95e-07 (expect ~1e-7, float32 storage precision -- see the same note in section 4; nothing double-counted or dropped in the sector mapping)


## 9. Save the output

In [10]:
out_cols = ["iso3", "country", "broad_sector", "year", "value_added_real",
            "employment", "employment_share", "labor_productivity_real",
            "labor_productivity_ppp"]
result = broad[out_cols].sort_values(["iso3", "year", "broad_sector"]).reset_index(drop=True)

out_path = OUT_DIR / "ggdc_africa_broad_sectors.csv"
result.to_csv(out_path, index=False)
print(f"Saved {len(result):,} rows -> {out_path}")
result.head(9)

Saved 3,327 rows -> ..\data\processed\ggdc_africa_broad_sectors.csv

,iso3,country,broad_sector,year,value_added_real,employment,employment_share,labor_productivity_real,labor_productivity_ppp
0,AGO,Angola,Agriculture,2002.0,2.617703e+05,2022.175000,0.369608,129.449896,1.248572
1,AGO,Angola,Manufacturing,2002.0,2.007762e+06,479.799000,0.087696,4184.590579,50.182770
2,AGO,Angola,Services,2002.0,1.763808e+06,2969.166054,0.542696,594.041548,5.576556
3,AGO,Angola,Agriculture,2003.0,2.827718e+05,2070.820000,0.365153,136.550618,1.384714
4,AGO,Angola,Manufacturing,2003.0,2.019986e+06,494.155000,0.087136,4087.758143,49.130448
5,AGO,Angola,Services,2003.0,1.854769e+06,3106.129905,0.547712,597.131819,6.228635
6,AGO,Angola,Agriculture,2004.0,3.084279e+05,2135.687000,0.361663,144.416249,1.662547
7,AGO,Angola,Manufacturing,2004.0,2.329146e+06,516.429000,0.087453,4510.098678,59.473866
8,AGO,Angola,Services,2004.0,1.994983e+06,3253.068044,0.550883,613.262072,7.142767


## 10. Limitations and next steps

- **Namibia (15 country-years) and Botswana (12 country-years)** have at least one
  missing sub-sector value in their earliest covered years; the affected
  `value_added_real`/`employment`/productivity cells are `NaN` here rather than
  silently `0` — see §5.
- **`Value_added_nominal` was not carried forward** — only real value added, since
  that (plus employment) is what the McMillan–Rodrik decomposition (methodology doc
  §6.3) and the requested employment shares actually need.
- **Employment units are inherited as-is from the source** (not independently
  re-verified here) — standard GGDC convention is thousands of persons, but confirm
  against the source's own documentation before reporting absolute levels.
- **Coverage varies sharply by country** (Algeria from 1999, Sierra Leone from 2001,
  most others from the 1960s–1970s) — a real constraint on which countries/years can
  enter the decomposition, not a cleaning artifact.
- **Next step**: merge `ggdc_africa_broad_sectors.csv` onto
  `data/processed/trade_agreements_with_exposure_country_year.csv` by `iso3`/`year`,
  per methodology doc §6.2 step 6, to build the full country–sector–year analysis
  panel the McMillan–Rodrik decomposition runs on.